# Notebook 2: Statistical Modeling
## March Madness 2026 Projection System
---
**Models:** Logistic Regression, XGBoost, Random Forest, Neural Network,
Bayesian Logistic Regression, Bayesian Hierarchical, Bayesian Neural Network, Beta-Binomial

**Key fix from v1:** Models now train on **team-level advanced stats** (SRS, eFG%, ORtg, Pace, etc.)
in addition to seed features. Features with partial coverage are imputed.

**Input:** `data/historical_features.csv`, `data/features_2026.csv`
**Output:** `models/` directory with trained model artifacts + `models/ensemble_config.json`

In [1]:
# ============================================================
# 2.0 CONFIG & IMPORTS
# ============================================================
import os, sys, json, warnings, logging, pickle, time
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("models")

DATA_DIR = Path("./data")
MODEL_DIR = Path("./models")
MODEL_DIR.mkdir(exist_ok=True)

# Reproducibility
RANDOM_SEED = 51
SALT = 2026_03
SEED = RANDOM_SEED + SALT
np.random.seed(SEED)

# M4 Optimization
import multiprocessing
N_CORES = multiprocessing.cpu_count()
N_JOBS = max(1, N_CORES - 1)
log.info(f"Config: seed={SEED}, cores={N_CORES}, n_jobs={N_JOBS}")

# Dependency check
DEPS = {}
def check_dep(name, import_name=None):
    try:
        mod = __import__(import_name or name)
        DEPS[name] = True
        return mod
    except ImportError:
        DEPS[name] = False
        log.info(f"  {name} not installed")
        return None

sklearn = check_dep("scikit-learn", "sklearn")
xgb_mod = check_dep("xgboost")
torch_mod = check_dep("pytorch", "torch")
pymc_mod = check_dep("pymc")

print("\nDependency status:")
for k, v in DEPS.items():
    print(f"  {'✅' if v else '❌'} {k}")
print(f"\nInstall missing: pip install scikit-learn xgboost torch pymc")

2026-03-18 21:56:31,154 [INFO] Config: seed=202654, cores=10, n_jobs=9
2026-03-18 21:56:33,297 [INFO] arviz_base not installed
2026-03-18 21:56:33,297 [INFO] arviz_stats not installed
2026-03-18 21:56:33,298 [INFO] arviz_plots not installed



Dependency status:
  ✅ scikit-learn
  ✅ xgboost
  ✅ pytorch
  ✅ pymc

Install missing: pip install scikit-learn xgboost torch pymc


In [2]:
# ============================================================
# 2.1 LOAD DATA & BUILD FEATURE MATRIX
# ============================================================
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

train_df = pd.read_csv(DATA_DIR / "historical_features.csv")
features_2026 = pd.read_csv(DATA_DIR / "features_2026.csv")

log.info(f"Historical matchups: {train_df.shape}")

# --- Feature groups ---
SEED_FEATURES = [
    "seed_diff", "seed_sum", "seed_diff_sq", "abs_seed_diff",
    "higher_seed_is_team1", "round_x_seeddiff", "is_upset_seed",
    "seed1_sq", "seed2_sq", "seed_product", "log_seed_ratio",
]
SREF_FEATURES = ["SRS_diff", "SOS_diff", "eFG%_diff", "TOV%_diff"]
SREF_ADV_FEATURES = ["ORtg_diff", "Pace_diff", "ORB%_diff"]
BT_FEATURES = ["AdjOE_diff", "AdjDE_diff", "Barthag_diff", "AdjTempo_diff", "WAB_diff"]
LEVEL_FEATURES = [
    "Pace_avg", "ORtg_avg", "SRS_sum",
    "AdjOE_avg", "AdjDE_avg", "AdjTempo_avg",
    "OE_vs_DE_1", "OE_vs_DE_2",
]

ALL_FEATURE_COLS = SEED_FEATURES.copy()
for group in [SREF_FEATURES, SREF_ADV_FEATURES, BT_FEATURES, LEVEL_FEATURES]:
    ALL_FEATURE_COLS.extend([f for f in group if f in train_df.columns])

log.info(f"Features ({len(ALL_FEATURE_COLS)}): {ALL_FEATURE_COLS}")

# Use 2008+ only (BT data starts here)
df_all = train_df[train_df["season"] >= 2008].copy()
df_train = df_all[df_all["season"] <= 2023]
df_val = df_all[df_all["season"] == 2024]
df_test = df_all[df_all["season"] == 2025]

X_train_raw = df_train[ALL_FEATURE_COLS].values
y_train = df_train["target"].values
X_val_raw = df_val[ALL_FEATURE_COLS].values
y_val = df_val["target"].values
X_test_raw = df_test[ALL_FEATURE_COLS].values
y_test = df_test["target"].values

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(imputer.fit_transform(X_train_raw))
X_val_sc = scaler.transform(imputer.transform(X_val_raw))
X_test_sc = scaler.transform(imputer.transform(X_test_raw))

df_full = df_all[df_all["season"] <= 2024]
X_full_raw = df_full[ALL_FEATURE_COLS].values
y_full = df_full["target"].values
X_full_sc = scaler.transform(imputer.transform(X_full_raw))

pickle.dump(scaler, open(MODEL_DIR / "scaler.pkl", "wb"))
pickle.dump(imputer, open(MODEL_DIR / "imputer.pkl", "wb"))
pickle.dump(ALL_FEATURE_COLS, open(MODEL_DIR / "feature_cols.pkl", "wb"))

log.info(f"Train: {X_train_sc.shape}, Val: {X_val_sc.shape}, Test: {X_test_sc.shape}")

# --- Rolling temporal CV for stable model comparison ---
CV_FOLDS = []
for val_year in range(2015, 2025):
    if val_year == 2020:
        continue
    tr = df_all[(df_all["season"] >= 2008) & (df_all["season"] < val_year)]
    va = df_all[df_all["season"] == val_year]
    if len(va) > 0:
        CV_FOLDS.append((tr, va, val_year))

log.info(f"Rolling CV: {len(CV_FOLDS)} folds")


def eval_model(name, y_true, probs, label=""):
    p = np.clip(probs, 0.001, 0.999)
    ll = log_loss(y_true, p)
    brier = brier_score_loss(y_true, p)
    auc = roc_auc_score(y_true, p)
    log.info(f"  {name} {label}: LL={ll:.4f}, Brier={brier:.4f}, AUC={auc:.4f}")
    return ll, brier, auc


def rolling_cv(fit_predict_fn, name):
    aucs, lls, briers = [], [], []
    for tr_df, va_df, val_year in CV_FOLDS:
        X_tr = scaler.transform(imputer.transform(tr_df[ALL_FEATURE_COLS].values))
        y_tr = tr_df["target"].values
        X_va = scaler.transform(imputer.transform(va_df[ALL_FEATURE_COLS].values))
        y_va = va_df["target"].values
        try:
            probs = fit_predict_fn(X_tr, y_tr, X_va)
            p = np.clip(probs, 0.001, 0.999)
            aucs.append(roc_auc_score(y_va, p))
            lls.append(log_loss(y_va, p))
            briers.append(brier_score_loss(y_va, p))
        except Exception as e:
            log.warning(f"  CV fold {val_year} failed: {e}")
    if aucs:
        log.info(f"  {name} CV: AUC={np.mean(aucs):.4f}+/-{np.std(aucs):.4f}, "
                 f"LL={np.mean(lls):.4f} ({len(aucs)} folds)")
    return np.mean(aucs) if aucs else 0.5, np.mean(lls) if lls else 1.0, np.mean(briers) if briers else 0.25


display(df_train[ALL_FEATURE_COLS].describe().round(3))

2026-03-18 21:56:34,249 [INFO] Historical matchups: (1448, 40)
2026-03-18 21:56:34,249 [INFO] Features (31): ['seed_diff', 'seed_sum', 'seed_diff_sq', 'abs_seed_diff', 'higher_seed_is_team1', 'round_x_seeddiff', 'is_upset_seed', 'seed1_sq', 'seed2_sq', 'seed_product', 'log_seed_ratio', 'SRS_diff', 'SOS_diff', 'eFG%_diff', 'TOV%_diff', 'ORtg_diff', 'Pace_diff', 'ORB%_diff', 'AdjOE_diff', 'AdjDE_diff', 'Barthag_diff', 'AdjTempo_diff', 'WAB_diff', 'Pace_avg', 'ORtg_avg', 'SRS_sum', 'AdjOE_avg', 'AdjDE_avg', 'AdjTempo_avg', 'OE_vs_DE_1', 'OE_vs_DE_2']
2026-03-18 21:56:34,254 [INFO] Train: (944, 31), Val: (63, 31), Test: (63, 31)
2026-03-18 21:56:34,256 [INFO] Rolling CV: 9 folds


,seed_diff,seed_sum,seed_diff_sq,abs_seed_diff,higher_seed_is_team1,round_x_seeddiff,is_upset_seed,seed1_sq,seed2_sq,seed_product,...,AdjTempo_diff,WAB_diff,Pace_avg,ORtg_avg,SRS_sum,AdjOE_avg,AdjDE_avg,AdjTempo_avg,OE_vs_DE_1,OE_vs_DE_2
count,944.000,944.000,944.000,944.000,944.000,944.000,944.000,944.000,944.000,944.000,...,941.000,941.000,762.000,762.000,873.000,941.000,941.000,941.000,941.000,941.000
mean,-4.039,13.632,59.024,6.423,0.730,-4.211,0.257,33.144,101.323,37.721,...,0.093,3.488,67.326,109.908,28.728,113.886,95.160,66.231,19.266,18.186
std,6.539,4.908,63.978,4.218,0.444,11.760,0.437,37.930,83.123,29.020,...,4.048,6.157,2.411,3.371,9.017,3.886,3.070,2.378,6.612,7.117
min,-15.000,2.000,0.000,0.000,0.000,-45.000,0.000,1.000,1.000,1.000,...,-11.608,-12.654,57.850,99.600,-5.840,103.582,86.594,58.216,0.521,-0.275
25%,-9.000,9.000,9.000,3.000,0.000,-12.000,0.000,4.000,16.000,14.000,...,-2.490,-0.735,65.650,107.800,22.480,111.168,93.011,64.593,14.643,13.626
50%,-4.000,17.000,49.000,7.000,1.000,-7.000,0.000,25.000,100.000,30.000,...,0.041,2.889,67.400,109.950,27.820,113.679,95.199,66.168,19.135,18.120
75%,1.000,17.000,81.000,9.000,1.000,2.000,1.000,49.000,169.000,60.000,...,2.883,7.142,68.950,112.050,35.440,116.327,97.320,68.010,23.736,22.912
max,10.000,25.000,225.000,15.000,1.000,50.000,1.000,256.000,256.000,156.000,...,14.676,25.304,74.350,120.300,53.330,128.666,105.132,73.035,41.356,44.394


In [3]:
# ============================================================
# 2.2 LOGISTIC REGRESSION
# ============================================================
from sklearn.linear_model import LogisticRegression

log.info("Training Logistic Regression...")

def lr_fit_predict(X_tr, y_tr, X_va):
    m = LogisticRegression(C=0.1, penalty="l2", max_iter=5000, random_state=SEED)
    m.fit(X_tr, y_tr)
    return m.predict_proba(X_va)[:, 1]

lr_cv_auc, lr_cv_ll, lr_cv_brier = rolling_cv(lr_fit_predict, "LogReg")

lr_model = LogisticRegression(C=0.1, penalty="l2", max_iter=5000, random_state=SEED)
lr_model.fit(X_train_sc, y_train)
lr_ll, lr_brier, lr_auc = eval_model("LogReg", y_val, lr_model.predict_proba(X_val_sc)[:, 1], "(VAL)")
eval_model("LogReg", y_test, lr_model.predict_proba(X_test_sc)[:, 1], "(TEST)")

lr_model.fit(X_full_sc, y_full)
pickle.dump(lr_model, open(MODEL_DIR / "logistic_regression.pkl", "wb"))

coefs = dict(zip(ALL_FEATURE_COLS, lr_model.coef_[0].round(4)))
log.info(f"  Top coefs: {dict(sorted(coefs.items(), key=lambda x: -abs(x[1]))[:6])}")

2026-03-18 21:56:34,274 [INFO] Training Logistic Regression...
2026-03-18 21:56:34,313 [INFO]   LogReg CV: AUC=0.8157+/-0.0500, LL=0.5046 (9 folds)
2026-03-18 21:56:34,317 [INFO]   LogReg (VAL): LL=0.4996, Brier=0.1699, AUC=0.8163
2026-03-18 21:56:34,318 [INFO]   LogReg (TEST): LL=0.3670, Brier=0.1171, AUC=0.9218
2026-03-18 21:56:34,321 [INFO]   Top coefs: {'AdjDE_diff': np.float64(-0.9201), 'AdjOE_diff': np.float64(0.8863), 'Barthag_diff': np.float64(0.3301), 'seed2_sq': np.float64(-0.2876), 'SOS_diff': np.float64(0.209), 'seed1_sq': np.float64(0.2073)}


In [5]:
# ============================================================
# 2.3 XGBOOST
# ============================================================
from xgboost import XGBClassifier

log.info("Training XGBoost...")

XGB_PARAMS = dict(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    reg_alpha=0.1, reg_lambda=1.0,
    objective="binary:logistic", eval_metric="logloss",
    random_state=SEED, n_jobs=N_JOBS, tree_method="hist",
)

def xgb_fit_predict(X_tr, y_tr, X_va):
    m = XGBClassifier(**XGB_PARAMS, early_stopping_rounds=50)
    m.fit(X_tr, y_tr, eval_set=[(X_va, np.zeros(len(X_va)))], verbose=False)
    return m.predict_proba(X_va)[:, 1]

# Note: rolling CV for XGB uses a dummy y_va for early stopping (not ideal).
# The holdout eval below is the primary XGB metric.
xgb_cv_auc, xgb_cv_ll, xgb_cv_brier = rolling_cv(xgb_fit_predict, "XGBoost")

# Holdout: proper early stopping with real val labels
xgb_model = XGBClassifier(**XGB_PARAMS, early_stopping_rounds=50)
xgb_model.fit(X_train_sc, y_train, eval_set=[(X_val_sc, y_val)], verbose=False)
best_n = xgb_model.best_iteration + 10
xgb_ll, xgb_brier, xgb_auc = eval_model("XGBoost", y_val, xgb_model.predict_proba(X_val_sc)[:, 1], "(VAL)")
eval_model("XGBoost", y_test, xgb_model.predict_proba(X_test_sc)[:, 1], "(TEST)")

# Retrain on full with fixed n_estimators
xgb_model = XGBClassifier(**{**XGB_PARAMS, "n_estimators": best_n})
xgb_model.fit(X_full_sc, y_full)
pickle.dump(xgb_model, open(MODEL_DIR / "xgboost.pkl", "wb"))

importance = dict(zip(ALL_FEATURE_COLS, xgb_model.feature_importances_.round(4)))
log.info(f"  Top features: {dict(sorted(importance.items(), key=lambda x: -x[1])[:6])}")

2026-03-18 21:57:50,012 [INFO] Training XGBoost...
2026-03-18 21:57:50,464 [INFO]   XGBoost CV: AUC=0.7482+/-0.0609, LL=0.6458 (9 folds)
2026-03-18 21:57:50,592 [INFO]   XGBoost (VAL): LL=0.5141, Brier=0.1707, AUC=0.8239
2026-03-18 21:57:50,594 [INFO]   XGBoost (TEST): LL=0.3985, Brier=0.1320, AUC=0.8866
2026-03-18 21:57:50,679 [INFO]   Top features: {'Barthag_diff': np.float32(0.1677), 'SRS_diff': np.float32(0.0611), 'WAB_diff': np.float32(0.0456), 'AdjOE_diff': np.float32(0.0399), 'AdjDE_diff': np.float32(0.0393), 'seed_sum': np.float32(0.0385)}


In [6]:
# ============================================================
# 2.4 RANDOM FOREST
# ============================================================
from sklearn.ensemble import RandomForestClassifier

log.info("Training Random Forest...")

RF_PARAMS = dict(
    n_estimators=1000, max_depth=6, min_samples_leaf=10,
    min_samples_split=20, max_features="sqrt",
    random_state=SEED, n_jobs=N_JOBS,
)

def rf_fit_predict(X_tr, y_tr, X_va):
    m = RandomForestClassifier(**RF_PARAMS)
    m.fit(X_tr, y_tr)
    return m.predict_proba(X_va)[:, 1]

rf_cv_auc, rf_cv_ll, rf_cv_brier = rolling_cv(rf_fit_predict, "RF")

rf_model = RandomForestClassifier(**RF_PARAMS, oob_score=True)
rf_model.fit(X_train_sc, y_train)
rf_ll, rf_brier, rf_auc = eval_model("RF", y_val, rf_model.predict_proba(X_val_sc)[:, 1], "(VAL)")
eval_model("RF", y_test, rf_model.predict_proba(X_test_sc)[:, 1], "(TEST)")
log.info(f"  OOB: {rf_model.oob_score_:.4f}")

rf_model = RandomForestClassifier(**RF_PARAMS)
rf_model.fit(X_full_sc, y_full)
pickle.dump(rf_model, open(MODEL_DIR / "random_forest.pkl", "wb"))

importance = dict(zip(ALL_FEATURE_COLS, rf_model.feature_importances_.round(4)))
log.info(f"  Top features: {dict(sorted(importance.items(), key=lambda x: -x[1])[:6])}")

2026-03-18 21:57:50,750 [INFO] Training Random Forest...
2026-03-18 21:57:55,553 [INFO]   RF CV: AUC=0.8020+/-0.0537, LL=0.5238 (9 folds)
2026-03-18 21:57:56,229 [INFO]   RF (VAL): LL=0.5154, Brier=0.1733, AUC=0.8087
2026-03-18 21:57:56,277 [INFO]   RF (TEST): LL=0.3880, Brier=0.1209, AUC=0.9172
2026-03-18 21:57:56,277 [INFO]   OOB: 0.7320
2026-03-18 21:57:56,836 [INFO]   Top features: {'Barthag_diff': np.float64(0.1587), 'SRS_diff': np.float64(0.112), 'WAB_diff': np.float64(0.0861), 'AdjOE_diff': np.float64(0.0803), 'AdjDE_diff': np.float64(0.0763), 'log_seed_ratio': np.float64(0.0528)}


In [7]:
# ============================================================
# 2.5 NEURAL NETWORK (PyTorch)
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

log.info("Training Neural Network...")
torch.manual_seed(SEED)

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
log.info(f"  Device: {device}")

n_features = X_train_sc.shape[1]
assert np.isfinite(X_train_sc).all(), "Non-finite in X_train_sc!"

class BracketNet(nn.Module):
    def __init__(self, n_feat, hidden=[64, 32, 16], dropout=0.3):
        super().__init__()
        layers = []
        prev = n_feat
        for h in hidden:
            layers.extend([nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)])
            prev = h
        self.features = nn.Sequential(*layers)
        self.head = nn.Linear(prev, 1)
        self.skip = nn.Linear(n_feat, prev)
    def forward(self, x):
        return torch.sigmoid(self.head(self.features(x) + 0.1 * self.skip(x)))


def train_nn(X_tr, y_tr, X_va=None, y_va=None, max_epochs=500, patience=30):
    model = BracketNet(X_tr.shape[1]).to(device)
    opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=200)
    crit = nn.BCELoss()
    X_t = torch.FloatTensor(X_tr).to(device)
    y_t = torch.FloatTensor(y_tr).unsqueeze(1).to(device)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=128, shuffle=True)
    
    best_loss, best_state, wait = float("inf"), None, 0
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            if X_va is not None:
                vl = crit(model(torch.FloatTensor(X_va).to(device)),
                          torch.FloatTensor(y_va).unsqueeze(1).to(device)).item()
            else:
                vl = crit(model(X_t), y_t).item()
        if vl < best_loss - 1e-5:
            best_loss = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                log.info(f"  NN early stop epoch {epoch}")
                break
    model.load_state_dict(best_state)
    model.eval()
    return model


def nn_fit_predict(X_tr, y_tr, X_va):
    m = train_nn(X_tr, y_tr, max_epochs=200, patience=20)
    with torch.no_grad():
        return m(torch.FloatTensor(X_va).to(device)).cpu().numpy().flatten()

nn_cv_auc, nn_cv_ll, nn_cv_brier = rolling_cv(nn_fit_predict, "NN")

nn_model = train_nn(X_train_sc, y_train, X_val_sc, y_val)
with torch.no_grad():
    nn_val_p = nn_model(torch.FloatTensor(X_val_sc).to(device)).cpu().numpy().flatten()
    nn_test_p = nn_model(torch.FloatTensor(X_test_sc).to(device)).cpu().numpy().flatten()
nn_ll, nn_brier, nn_auc = eval_model("NN", y_val, nn_val_p, "(VAL)")
eval_model("NN", y_test, nn_test_p, "(TEST)")

# Retrain on full
nn_model = train_nn(X_full_sc, y_full, max_epochs=150, patience=30)
torch.save(nn_model.state_dict(), MODEL_DIR / "neural_net.pt")
pickle.dump({"n_features": n_features, "hidden_dims": [64, 32, 16]}, open(MODEL_DIR / "nn_config.pkl", "wb"))

2026-03-18 21:57:56,842 [INFO] Training Neural Network...
2026-03-18 21:57:56,904 [INFO]   Device: mps
2026-03-18 21:58:06,024 [INFO]   NN early stop epoch 195
2026-03-18 21:58:15,722 [INFO]   NN early stop epoch 190
2026-03-18 21:58:27,739 [INFO]   NN CV: AUC=0.7903+/-0.0426, LL=0.5867 (9 folds)
2026-03-18 21:58:28,900 [INFO]   NN early stop epoch 52
2026-03-18 21:58:28,904 [INFO]   NN (VAL): LL=0.4946, Brier=0.1690, AUC=0.8261
2026-03-18 21:58:28,905 [INFO]   NN (TEST): LL=0.3728, Brier=0.1240, AUC=0.8980


In [8]:
# ============================================================
# 2.6 BAYESIAN LOGISTIC REGRESSION (PyMC)
# ============================================================
import pymc as pm
import arviz as az

log.info("Training Bayesian Logistic Regression...")

with pm.Model() as bayes_lr_model:
    alpha = pm.Normal("alpha", mu=0, sigma=2)
    betas = pm.Normal("betas", mu=0, sigma=1, shape=X_train_sc.shape[1])
    mu = alpha + pm.math.dot(X_train_sc, betas)
    p = pm.Deterministic("p", pm.math.sigmoid(mu))
    y_obs = pm.Bernoulli("y_obs", p=p, observed=y_train)

    trace_lr = pm.sample(
        draws=2000, tune=1000, chains=4,
        cores=min(4, N_CORES), target_accept=0.9,
        random_seed=SEED, return_inferencedata=True,
    )

posterior_betas = trace_lr.posterior["betas"].values
posterior_alpha = trace_lr.posterior["alpha"].values
beta_mean = posterior_betas.mean(axis=(0, 1))
alpha_mean = posterior_alpha.mean()
blr_train_logits = alpha_mean + X_train_sc @ beta_mean
blr_train_probs = 1 / (1 + np.exp(-blr_train_logits))
eval_model("BayesLR", y_train, blr_train_probs, "(train)")

blr_val_logits = alpha_mean + X_val_sc @ beta_mean
blr_val_probs = 1 / (1 + np.exp(-blr_val_logits))
blr_ll, blr_brier, blr_auc = eval_model("BayesLR", y_val, blr_val_probs, "(VAL)")

blr_test_logits = alpha_mean + X_test_sc @ beta_mean
blr_test_probs = 1 / (1 + np.exp(-blr_test_logits))
eval_model("BayesLR", y_test, blr_test_probs, "(TEST)")

# Full data probs for ensemble
blr_probs = 1 / (1 + np.exp(-(alpha_mean + X_full_sc @ beta_mean)))

trace_lr.to_netcdf(str(MODEL_DIR / "bayesian_lr_trace.nc"))

summary = az.summary(trace_lr, var_names=["betas"])
summary.index = ALL_FEATURE_COLS
print("\nPosterior coefficient summaries:")
display(summary[["mean", "sd", "hdi_3%", "hdi_97%"]])

2026-03-18 21:58:32,325 [INFO] Training Bayesian Logistic Regression...
2026-03-18 21:58:32,401 [INFO] Initializing NUTS using jitter+adapt_diag...
2026-03-18 21:58:34,479 [INFO] Multiprocess sampling (4 chains in 4 jobs)
2026-03-18 21:58:34,479 [INFO] NUTS: [alpha, betas]


Output()

2026-03-18 21:58:49,006 [INFO] Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 15 seconds.
2026-03-18 21:58:49,180 [INFO]   BayesLR (train): LL=0.4777, Brier=0.1591, AUC=0.8357
2026-03-18 21:58:49,181 [INFO]   BayesLR (VAL): LL=0.5142, Brier=0.1741, AUC=0.8120
2026-03-18 21:58:49,183 [INFO]   BayesLR (TEST): LL=0.3349, Brier=0.1064, AUC=0.9286



Posterior coefficient summaries:


,mean,sd,hdi_3%,hdi_97%
seed_diff,0.538,0.790,-0.939,2.021
seed_sum,0.468,0.435,-0.339,1.294
seed_diff_sq,-0.133,0.592,-1.279,0.934
abs_seed_diff,-0.220,0.324,-0.840,0.378
higher_seed_is_team1,0.326,0.313,-0.283,0.892
round_x_seeddiff,0.085,0.228,-0.347,0.501
is_upset_seed,0.223,0.296,-0.331,0.782
seed1_sq,0.060,0.451,-0.827,0.876
seed2_sq,-0.375,0.760,-1.798,1.044
seed_product,-0.343,0.545,-1.343,0.685


In [14]:
# ============================================================
# 2.7 BAYESIAN HIERARCHICAL MODEL
# ============================================================
import pymc as pm
import arviz as az

log.info("Training Bayesian Hierarchical Model...")

def seed_group(s):
    if s <= 2: return 0
    if s <= 4: return 1
    if s <= 8: return 2
    if s <= 12: return 3
    return 4

seed1_groups = np.array([seed_group(s) for s in df_train["seed1"].values])
n_groups = 5

with pm.Model() as hier_model:
    # Hyperpriors
    mu_alpha = pm.Normal("mu_alpha", mu=0, sigma=2)
    sigma_alpha = pm.HalfNormal("sigma_alpha", sigma=1)
    mu_beta = pm.Normal("mu_beta", mu=0, sigma=1)
    sigma_beta = pm.HalfNormal("sigma_beta", sigma=0.5)

    # Non-centered parameterization (fixes divergences)
    alpha_offset = pm.Normal("alpha_offset", mu=0, sigma=1, shape=5)
    alpha_group = pm.Deterministic("alpha_group", mu_alpha + sigma_alpha * alpha_offset)
    
    beta_offset = pm.Normal("beta_offset", mu=0, sigma=1, shape=5)
    beta_seed_diff = pm.Deterministic("beta_seed_diff", mu_beta + sigma_beta * beta_offset)
    
    beta_global = pm.Normal("beta_global", mu=0, sigma=1, shape=X_train_sc.shape[1])

    game_alpha = alpha_group[seed1_groups]
    game_beta_sd = beta_seed_diff[seed1_groups]
    mu = game_alpha + game_beta_sd * df_train["seed_diff"].values + pm.math.dot(X_train_sc, beta_global)

    p = pm.Deterministic("p", pm.math.sigmoid(mu))
    y_obs = pm.Bernoulli("y_obs", p=p, observed=y_train)

    trace_hier = pm.sample(
        draws=2000, tune=3000, chains=4,
        cores=min(4, N_CORES), target_accept=0.99,
        nuts_sampler_kwargs={"max_treedepth": 15},
        random_seed=SEED, return_inferencedata=True,
    )

hier_summary = az.summary(trace_hier, var_names=["alpha_group", "beta_seed_diff", "mu_alpha", "mu_beta"])
print("\nHierarchical model posterior:")
display(hier_summary)
trace_hier.to_netcdf(str(MODEL_DIR / "bayesian_hier_trace.nc"))
log.info("  Bayesian Hierarchical model trained successfully")

2026-03-18 22:12:59,637 [INFO] Training Bayesian Hierarchical Model...
2026-03-18 22:12:59,774 [INFO] Initializing NUTS using jitter+adapt_diag...
2026-03-18 22:13:03,550 [INFO] Multiprocess sampling (4 chains in 4 jobs)
2026-03-18 22:13:03,551 [INFO] NUTS: [mu_alpha, sigma_alpha, mu_beta, sigma_beta, alpha_offset, beta_offset, beta_global]


Output()

2026-03-18 22:14:43,844 [INFO] Sampling 4 chains for 3_000 tune and 2_000 draw iterations (12_000 + 8_000 draws total) took 100 seconds.



Hierarchical model posterior:


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
alpha_group[0],1.244,0.964,-0.602,3.046,0.015,0.010,4061.0,5248.0,1.0
alpha_group[1],1.398,0.952,-0.411,3.200,0.015,0.010,4023.0,5126.0,1.0
alpha_group[2],1.362,0.898,-0.400,2.995,0.014,0.009,4221.0,4961.0,1.0
alpha_group[3],1.553,0.968,-0.293,3.327,0.014,0.010,4562.0,5737.0,1.0
alpha_group[4],1.190,1.093,-0.780,3.302,0.016,0.013,4478.0,4870.0,1.0
beta_seed_diff[0],0.116,0.223,-0.281,0.559,0.004,0.002,3970.0,5030.0,1.0
beta_seed_diff[1],0.125,0.221,-0.292,0.539,0.004,0.002,3948.0,4791.0,1.0
beta_seed_diff[2],0.140,0.220,-0.255,0.568,0.003,0.002,3989.0,5148.0,1.0
beta_seed_diff[3],0.091,0.228,-0.353,0.508,0.004,0.002,3899.0,5015.0,1.0
beta_seed_diff[4],0.047,0.291,-0.437,0.619,0.005,0.005,3578.0,4629.0,1.0


2026-03-18 22:14:45,463 [INFO]   Bayesian Hierarchical model trained successfully


In [15]:
# ============================================================
# 2.8 GAUSSIAN PROCESS CLASSIFICATION — SKIPPED
# ============================================================
# GP is O(n³) and impractical for ~1,400 samples without heavy subsampling.
# The ensemble has 7 other models including 3 Bayesian methods.
log.info("Skipping GP Classification (computational cost too high)")
gp_model = None
gp_probs = None
GP_BACKEND = None

2026-03-18 22:14:45,466 [INFO] Skipping GP Classification (computational cost too high)


In [16]:
# ============================================================
# 2.9 BAYESIAN NEURAL NETWORK
# ============================================================
import pymc as pm

log.info("Training Bayesian Neural Network...")

n_hidden = 16

with pm.Model() as bnn_model:
    w1 = pm.Normal("w1", mu=0, sigma=1, shape=(X_train_sc.shape[1], n_hidden))
    b1 = pm.Normal("b1", mu=0, sigma=1, shape=n_hidden)
    h1 = pm.math.tanh(pm.math.dot(X_train_sc, w1) + b1)

    w2 = pm.Normal("w2", mu=0, sigma=1, shape=(n_hidden, n_hidden))
    b2 = pm.Normal("b2", mu=0, sigma=1, shape=n_hidden)
    h2 = pm.math.tanh(pm.math.dot(h1, w2) + b2)

    w_out = pm.Normal("w_out", mu=0, sigma=1, shape=(n_hidden, 1))
    b_out = pm.Normal("b_out", mu=0, sigma=1)
    logits = pm.math.dot(h2, w_out).flatten() + b_out

    p = pm.Deterministic("p", pm.math.sigmoid(logits))
    y_obs = pm.Bernoulli("y_obs", p=p, observed=y_train)

    log.info("  Using ADVI approximation...")
    approx = pm.fit(n=30000, method="advi", random_seed=SEED)
    trace_bnn = approx.sample(2000)

bnn_p_samples = trace_bnn.posterior["p"].values
bnn_probs = bnn_p_samples.mean(axis=(0, 1))
bnn_probs_std = bnn_p_samples.std(axis=(0, 1))

# Validate BNN predictions are reasonable
bnn_ll = log_loss(y_train, np.clip(bnn_probs, 0.01, 0.99))
bnn_brier = brier_score_loss(y_train, np.clip(bnn_probs, 0.01, 0.99))
bnn_auc = roc_auc_score(y_train, bnn_probs)

log.info(f"  BNN - LogLoss: {bnn_ll:.4f}, Brier: {bnn_brier:.4f}, AUC: {bnn_auc:.4f}")
log.info(f"  Mean uncertainty (std): {bnn_probs_std.mean():.4f}")

# Flag if BNN is broken (AUC < 0.5 means inverted predictions)
if bnn_auc < 0.5:
    log.warning(f"  ⚠️ BNN AUC={bnn_auc:.4f} < 0.5 — predictions are inverted or degenerate.")
    log.warning(f"  BNN will be EXCLUDED from ensemble.")
    bnn_probs = None  # exclude from ensemble

np.savez(MODEL_DIR / "bnn_results.npz", probs_mean=bnn_probs if bnn_probs is not None else np.array([]),
         probs_std=bnn_probs_std)

2026-03-18 22:14:45,470 [INFO] Training Bayesian Neural Network...
2026-03-18 22:14:45,478 [INFO]   Using ADVI approximation...


Output()

2026-03-18 22:14:58,610 [INFO] Finished [100%]: Average Loss = 676.18
2026-03-18 22:14:59,105 [INFO]   BNN - LogLoss: 0.6627, Brier: 0.2349, AUC: 0.5933
2026-03-18 22:14:59,105 [INFO]   Mean uncertainty (std): 0.0870


In [17]:
# ============================================================
# 2.10 BETA-BINOMIAL SEED MODEL
# ============================================================
from scipy.stats import beta as beta_dist

log.info("Training Beta-Binomial seed model...")

matchup_counts = {}
for _, row in df_train.iterrows():
    s1, s2 = int(row["seed1"]), int(row["seed2"])
    key = (min(s1, s2), max(s1, s2))
    if key not in matchup_counts:
        matchup_counts[key] = {"wins_lower": 0, "total": 0}
    matchup_counts[key]["total"] += 1
    if (s1 < s2 and row["target"] == 1) or (s1 > s2 and row["target"] == 0):
        matchup_counts[key]["wins_lower"] += 1

beta_binomial_params = {}
prior_alpha, prior_beta = 2.0, 2.0

for (s1, s2), counts in sorted(matchup_counts.items()):
    wins = counts["wins_lower"]
    total = counts["total"]
    post_alpha = prior_alpha + wins
    post_beta_val = prior_beta + (total - wins)
    mean_p = post_alpha / (post_alpha + post_beta_val)
    std_p = np.sqrt(post_alpha * post_beta_val / ((post_alpha + post_beta_val)**2 * (post_alpha + post_beta_val + 1)))

    beta_binomial_params[(s1, s2)] = {
        "alpha": post_alpha, "beta": post_beta_val,
        "mean": mean_p, "std": std_p,
        "n_games": total, "wins_lower": wins,
    }
    log.info(f"  {s1:>2} vs {s2:>2}: {mean_p:.3f} ± {std_p:.3f} ({wins}/{total})")

pickle.dump(beta_binomial_params, open(MODEL_DIR / "beta_binomial.pkl", "wb"))
log.info("  Beta-Binomial model complete")

2026-03-18 22:14:59,109 [INFO] Training Beta-Binomial seed model...
2026-03-18 22:14:59,121 [INFO]    1 vs  1: 0.167 ± 0.103 (0/8)
2026-03-18 22:14:59,121 [INFO]    1 vs  2: 0.538 ± 0.096 (12/22)
2026-03-18 22:14:59,121 [INFO]    1 vs  3: 0.750 ± 0.094 (13/16)
2026-03-18 22:14:59,121 [INFO]    1 vs  4: 0.733 ± 0.079 (20/26)
2026-03-18 22:14:59,121 [INFO]    1 vs  5: 0.636 ± 0.100 (12/18)
2026-03-18 22:14:59,122 [INFO]    1 vs  6: 0.667 ± 0.178 (2/2)
2026-03-18 22:14:59,122 [INFO]    1 vs  7: 0.571 ± 0.175 (2/3)
2026-03-18 22:14:59,122 [INFO]    1 vs  8: 0.757 ± 0.070 (26/33)
2026-03-18 22:14:59,122 [INFO]    1 vs  9: 0.839 ± 0.065 (24/27)
2026-03-18 22:14:59,122 [INFO]    1 vs 10: 0.625 ± 0.161 (3/4)
2026-03-18 22:14:59,122 [INFO]    1 vs 11: 0.556 ± 0.157 (3/5)
2026-03-18 22:14:59,122 [INFO]    1 vs 12: 0.818 ± 0.111 (7/7)
2026-03-18 22:14:59,123 [INFO]    1 vs 13: 0.600 ± 0.200 (1/1)
2026-03-18 22:14:59,123 [INFO]    1 vs 16: 0.938 ± 0.030 (58/60)
2026-03-18 22:14:59,123 [INFO]    2 

In [18]:
# ============================================================
# 2.11 ENSEMBLE & MODEL SUMMARY
# ============================================================
log.info("Building ensemble...")

model_results = {
    "LogReg":       {"cv_auc": lr_cv_auc, "val_auc": lr_auc, "val_ll": lr_ll, "val_brier": lr_brier},
    "XGBoost":      {"cv_auc": xgb_cv_auc, "val_auc": xgb_auc, "val_ll": xgb_ll, "val_brier": xgb_brier},
    "RandomForest": {"cv_auc": rf_cv_auc, "val_auc": rf_auc, "val_ll": rf_ll, "val_brier": rf_brier},
    "NeuralNet":    {"cv_auc": nn_cv_auc, "val_auc": nn_auc, "val_ll": nn_ll, "val_brier": nn_brier},
    "BayesLR":      {"cv_auc": 0, "val_auc": blr_auc, "val_ll": blr_ll, "val_brier": blr_brier},
}

# Weights by CV AUC^2 (BayesLR uses val_auc as fallback since no CV)
weights = {}
for name, res in model_results.items():
    auc = res["cv_auc"] if res["cv_auc"] > 0.5 else res["val_auc"]
    weights[name] = auc ** 2
total_w = sum(weights.values())
weights = {k: v / total_w for k, v in weights.items()}

log.info("\nEnsemble weights (CV AUC^2):")
for name, w in sorted(weights.items(), key=lambda x: -x[1]):
    cv = model_results[name]["cv_auc"]
    val = model_results[name]["val_auc"]
    log.info(f"  {name:15s}: {w:.4f} (CV={cv:.4f}, Val={val:.4f})")

# Evaluate ensemble on val using the FINAL retrained models
ens_val_probs = np.zeros(len(y_val))
for name in weights:
    if name == "LogReg":
        p = lr_model.predict_proba(X_val_sc)[:, 1]
    elif name == "XGBoost":
        p = xgb_model.predict_proba(X_val_sc)[:, 1]
    elif name == "RandomForest":
        p = rf_model.predict_proba(X_val_sc)[:, 1]
    elif name == "NeuralNet":
        nn_model.eval()
        with torch.no_grad():
            p = nn_model(torch.FloatTensor(X_val_sc).to(device)).cpu().numpy().flatten()
    elif name == "BayesLR":
        beta_mean = trace_lr.posterior["betas"].values.mean(axis=(0, 1))
        alpha_mean = trace_lr.posterior["alpha"].values.mean()
        p = 1 / (1 + np.exp(-(alpha_mean + X_val_sc @ beta_mean)))
    else:
        continue
    ens_val_probs += weights[name] * np.clip(p, 0.001, 0.999)

ens_ll, ens_brier, ens_auc = eval_model("ENSEMBLE", y_val, ens_val_probs, "(VAL)")

results = {
    "weights": weights,
    "metrics": {k: {kk: round(vv, 4) for kk, vv in v.items()} for k, v in model_results.items()},
    "feature_cols": ALL_FEATURE_COLS,
    "ensemble_logloss": round(ens_ll, 4), "ensemble_auc": round(ens_auc, 4),
    "seed": SEED, "timestamp": datetime.now().isoformat(),
}
with open(MODEL_DIR / "ensemble_config.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

comp_df = pd.DataFrame([
    {"Model": name, "CV_AUC": round(res["cv_auc"], 4), "Val_AUC": round(res["val_auc"], 4),
     "Val_LL": round(res["val_ll"], 4), "Val_Brier": round(res["val_brier"], 4),
     "Weight": round(weights[name], 4)}
    for name, res in model_results.items()
]).sort_values("Val_LL")

print("\n Model Comparison:")
display(comp_df)
comp_df.to_csv(MODEL_DIR / "model_comparison.csv", index=False)

2026-03-18 22:14:59,146 [INFO] Building ensemble...
2026-03-18 22:14:59,148 [INFO] 
Ensemble weights (CV AUC^2):
2026-03-18 22:14:59,148 [INFO]   LogReg         : 0.2111 (CV=0.8157, Val=0.8163)
2026-03-18 22:14:59,149 [INFO]   BayesLR        : 0.2092 (CV=0.0000, Val=0.8120)
2026-03-18 22:14:59,149 [INFO]   RandomForest   : 0.2040 (CV=0.8020, Val=0.8087)
2026-03-18 22:14:59,150 [INFO]   NeuralNet      : 0.1981 (CV=0.7903, Val=0.8261)
2026-03-18 22:14:59,150 [INFO]   XGBoost        : 0.1776 (CV=0.7482, Val=0.8239)
2026-03-18 22:14:59,220 [INFO]   ENSEMBLE (VAL): LL=0.4001, Brier=0.1300, AUC=0.8957



 Model Comparison:


,Model,CV_AUC,Val_AUC,Val_LL,Val_Brier,Weight
3,NeuralNet,0.7903,0.8261,0.4946,0.1690,0.1981
0,LogReg,0.8157,0.8163,0.4996,0.1699,0.2111
1,XGBoost,0.7482,0.8239,0.5141,0.1707,0.1776
4,BayesLR,0.0000,0.8120,0.5142,0.1741,0.2092
2,RandomForest,0.8020,0.8087,0.5154,0.1733,0.2040


## Modeling Summary

**Key improvement:** Models now train on **18 features** including SRS_diff, SOS_diff, eFG%_diff, TOV%_diff, ORtg_diff, Pace_diff, ORB%_diff — not just seed-based features.

**Models saved to `./models/`:**
1. Logistic Regression — interpretable baseline with advanced stats
2. XGBoost — non-linear interactions between seed + efficiency features
3. Random Forest — robust ensemble, OOB validation
4. Neural Network — complex patterns, MPS-accelerated
5. Bayesian LR — full posterior with uncertainty
6. Bayesian Hierarchical — partial pooling across seed tiers
7. BNN — weight uncertainty (excluded if AUC < 0.5)
8. Beta-Binomial — conjugate seed matchup model

**Next:** Open `03_simulation.ipynb`